# W33 GF(3) QEC demo

This notebook demonstrates the ternary (GF(3)) encoder/decoder and the
MLUT (table-driven) decoder implemented in `scripts/w33_quantum_error_correction.py`.

- Build the code from W33 triangle/edge incidence
- Show min distance and correction radius t
- Encode/decode examples and Monte‑Carlo error statistics


In [ ]:
# Environment & imports
import sys
sys.path.insert(0, "scripts")

import numpy as np
import pprint

from w33_homology import build_clique_complex, boundary_matrix, build_w33
from scripts.w33_quantum_error_correction import (
    compute_basis_rows_mod3,
    encode_message,
    decode_message,
    build_mlut_table,
    decode_via_mlut,
    code_min_distance_from_basis,
    analyze_w33_qec,
)

np.random.seed(42)

In [ ]:
# Build code from W33 and show code parameters
n, vertices, adj, edges = build_w33()
simplices = build_clique_complex(n, adj)
B2 = boundary_matrix(simplices[2], simplices[1]).astype(int)
M = B2.T

basis = compute_basis_rows_mod3(M)
print('basis shape (k x n):', basis.shape)
min_d = code_min_distance_from_basis(basis)
print('code min distance d =', min_d)
print('correction radius t =', (min_d - 1)//2 if min_d>0 else 0)

pprint.pprint(analyze_w33_qec())

In [ ]:
# Encode / single-error decode example
k = basis.shape[0]
msg = np.zeros(k, dtype=int)
if k >= 3:
    msg[:3] = [1, 2, 1]

cw = encode_message(basis, msg)
print('message (first 8):', msg[:8])
print('codeword (first 16):', cw[:16])

# single-symbol corruption
recv = cw.copy()
recv[5] = (recv[5] + 1) % 3
print('\nCorrupted position 5 ->', recv[:16])
dec_msg, corrected_cw, ok = decode_message(recv, basis)
print('decode_message ok=', ok)
print('recovered message equals original?', np.array_equal(dec_msg % 3, msg % 3))

# MLUT decode on same corruption
mlut, t = build_mlut_table(basis)
dec2_msg, corr2, ok2 = decode_via_mlut(recv, basis, mlut=mlut)
print('decode_via_mlut ok=', ok2)
print('mlut t=', t)

In [ ]:
# Monte Carlo: single-symbol errors vs MLUT radius-correction
import numpy as np
rng = np.random.default_rng(2026)

# single-symbol correction success (syndrome decoder)
N = 300
succ_single = 0
succ_mlut = 0
for _ in range(N):
    msg = rng.integers(0, 3, size=k, dtype=int)
    cw = encode_message(basis, msg)
    pos = rng.integers(0, cw.size)
    val = int(rng.choice([1, 2]))
    recv = cw.copy()
    recv[pos] = int((recv[pos] + val) % 3)
    dm, _, ok = decode_message(recv, basis)
    if ok and np.array_equal(dm % 3, msg % 3):
        succ_single += 1
    dm2, _, ok2 = decode_via_mlut(recv, basis, mlut=mlut)
    if ok2 and np.array_equal(dm2 % 3, msg % 3):
        succ_mlut += 1

print(f"single-error decoder success rate: {succ_single}/{N}")
print(f"mlut decoder success rate:        {succ_mlut}/{N} (should match for single errors)")

# errors up to radius t: sample and measure mlut success
if t > 0:
    trials = 200
    succ_up_to_t = 0
    for _ in range(trials):
        msg = rng.integers(0, 3, size=k, dtype=int)
        cw = encode_message(basis, msg)
        w = rng.integers(1, t + 1)
        pos = rng.choice(range(cw.size), size=w, replace=False)
        vals = rng.integers(1, 3, size=w)
        recv = cw.copy()
        for p, v in zip(pos, vals):
            recv[p] = int((recv[p] + v) % 3)
        dm, _, ok = decode_via_mlut(recv, basis, mlut=mlut)
        if ok and np.array_equal(dm % 3, msg % 3):
            succ_up_to_t += 1
    print(f"mlut success rate for random errors of weight<=t: {succ_up_to_t}/{trials}")
else:
    print('t == 0: no multi-error correction radius available')

# Short conclusion

- The W33-derived ternary code has a nontrivial minimum distance (see earlier cell).
- The single-symbol syndrome decoder and the MLUT decoder both recover single-symbol errors.
- The MLUT (table lookup) extends correction up to the code radius t (when table is built up to that weight).

Explore the other functions in `scripts/w33_quantum_error_correction.py` for further QEC primitives.

In [ ]:
# Visualize MLUT coverage & correction-weight distribution
import matplotlib.pyplot as plt
from scripts.w33_quantum_error_correction import mlut_coverage_stats, build_approx_mlut_table

# build (or sample) MLUT and compute coverage
mlut, t_sample, coverage = build_approx_mlut_table(basis, max_weight=max(1, t), max_entries=5000)
stats = mlut_coverage_stats(mlut, basis)
print('MLUT table size:', stats['table_size'])
print('Syndrome space:', stats['total_syndromes'])
print('Coverage fraction:', f"{stats['coverage_fraction']:.4f}")
print('Correction-weight histogram:', stats['weight_hist'])

# histogram of correction weights
weights = []
for e in mlut.values():
    weights.append(int(np.count_nonzero(e)))

plt.figure(figsize=(6, 3))
plt.hist(weights, bins=range(0, max(weights) + 2), align='left')
plt.xlabel('Correction weight')
plt.ylabel('Count (MLUT entries)')
plt.title('MLUT correction-weight distribution')
plt.grid(alpha=0.3)
plt.show()